# fMRI Reliability Pipeline
- Designed to parametrically analyze fMRI files to identify which pipelines produce the most robust results.

In [ ]:
import pandas as pd
import itertools
import numpy as np

from scipy.spatial.distance import pdist
from scipy.stats import f_oneway, ttest_ind

# Plotting Packages 
import seaborn as sns
import matplotlib.pyplot as plt
import ptitprince as pt # Only needed for raincloud plot
import configFunctions as config
from nilearn import plotting


### Required files
- 'coordinatesAll.csv' - generated by serverFunctions, final output of mergeOutputs.sh.
- 

In [ ]:
# A configuration function to make output images easier to modify for publication.
config.svg_editing()

In [ ]:
# load data, inspect
df = pd.read_csv('coordinatesAll.csv')
df.head()

### Intermediate clean up

In [ ]:
# Pull out and show the rows which include any nans
rows_with_nan = df[df.isna().any(axis=1)]
rows_with_nan.to_csv('coordinatesAll_NaNRows.csv', index=False)
subj_with_nan = rows_with_nan['subject'].unique()
print(f'Subjects with NaNs - {len(subj_with_nan)}')
print(f'List: {subj_with_nan}')
print(f'Dataframe filtered to remove subjects with any NaNs')

df = df[~df['subject'].isin(subj_with_nan)]
# ['107','103','99','45','83','108','110','88','109','115','111','15','116','112','104','100','117','105','101','118','102']
print(f'{len(df['subject'].unique())} Subjects Remaining...')


In [ ]:
# Group by subject and condition
grouping_vars = ['subject', 'target', 'clustMask', 'sequence_type']

subjVec = df['subject'].unique()
targetVec = df['target'].unique()
cMaskVec = df['clustMask'].unique()
seqTypeVec = df['sequence_type'].unique()

# Create an array with each possible combo.
combinations = list(itertools.product(subjVec, targetVec, cMaskVec, seqTypeVec))

# Put into pandas dataframe
combodf = pd.DataFrame(combinations, columns=['subject', 'target', 'clustMask', 'sequence_type'])
combodf.head()

In [ ]:
# Perform the analysis - span the set of each combination of parameters, take the mean euclidean distance across those sessions.
results = []

for combo in combinations:
    df_filt = df[
        (df['subject'] == combo[0]) & 
        (df['target'] == combo[1]) & 
        (df['clustMask'] == combo[2]) & 
        (df['sequence_type'] == combo[3])
    ]
    meanDist = round(pdist(df_filt[['X','Y','Z']], metric='euclidean').mean(),2)
    results.append(meanDist)

# Append this new column to the previously generated df.
combodf['mean_distance'] = results
combodf.head()

# Plotting

In [ ]:
groupVars = ['target', 'clustMask', 'sequence_type'] # Matching df column names
targetName = 'sgACC' # For plotting, what does the 'target' mask represent?
clusterRegion = 'dlPFC' # for plotting, what does the 'clusterMask' mask represent?
groupVarNames = [f'{targetName} mask', f'{clusterRegion} mask', 'Sequence Type'] # The names for each of the above groupVars for plotting purposes

# For nilearn brain views, to visualize the output coordinates of processing with the clustMask type mentioned below, must match exactly.
colorDict = {
    'orig': 'red',
    'dilate_5': 'magenta',
    'erode_1': 'blue'
}


### Single Subject plot

In [ ]:
df_subj = df[(df['subject'] == 1) & (df['session'] == 1)].copy()
df_subj['colors'] = df_subj['clustMask'].map(colorDict)

plot_coords = np.array(df_subj[['X','Y','Z']])
plot_colors = list(df_subj['colors'])
view = plotting.view_markers(plot_coords, plot_colors, marker_size=5)

# Viewing options
# view.open_in_browser()
view

### Plot the mean coordinates for each subject

In [ ]:
df_subj_centroid = df.groupby('subject')[['X', 'Y', 'Z']].mean()

df_subj['colors'] = df_subj['clustMask'].map(colorDict)

plot_coords = np.array(df_subj_centroid[['X','Y','Z']])
# plot_colors = list(df_subj['colors'])
view = plotting.view_markers(plot_coords, 'red', marker_size=5)
# view.open_in_browser()
view

### Traditional Violin + Rain plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (var, varPlot) in enumerate(zip(groupVars, groupVarNames)):
    # Violin plot
    sns.violinplot(data=combodf, x=var, y='mean_distance', ax=axes[idx], 
                   inner=None, alpha=0.6)
    
    # Overlay individual data points
    sns.stripplot(data=combodf, x=var, y='mean_distance', ax=axes[idx],
                  color='black', alpha=0.4, size=3, jitter=True)
    
    # Get unique groups
    groups = combodf[var].unique()
    group_data = [combodf[combodf[var] == val]['mean_distance'].dropna()
                  for val in groups]
    
    # Perform statistical test (t-test for 2 groups, ANOVA for more)
    if len(groups) == 2:
        stat, p_val = ttest_ind(group_data[0], group_data[1])
        stat_name = 't'
        
        # Add significance line and star
        y_max = combodf['mean_distance'].max()
        y_range = combodf['mean_distance'].max() - combodf['mean_distance'].min()
        line_height = y_max + 0.05 * y_range
        star_height = line_height + 0.02 * y_range
        
        # Draw line
        axes[idx].plot([0, 1], [line_height, line_height], 'k-', linewidth=1.5)
        axes[idx].plot([0, 0], [line_height - 0.01 * y_range, line_height], 'k-', linewidth=1.5)
        axes[idx].plot([1, 1], [line_height - 0.01 * y_range, line_height], 'k-', linewidth=1.5)
        
        # Add star based on significance
        if p_val < 0.001:
            star = '***'
        elif p_val < 0.01:
            star = '**'
        elif p_val < 0.05:
            star = '*'
        else:
            star = 'ns'
        
        axes[idx].text(0.5, star_height, star, ha='center', va='bottom', 
                      fontsize=14, fontweight='bold')
    else:
        # ANOVA for more than 2 groups
        stat, p_val = f_oneway(*group_data)
        stat_name = 'F'
    
    # Add title with statistics
    axes[idx].set_title(f'{varPlot}\n{stat_name}={stat:.2f}, p={p_val:.4f}',
                        fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('Mean Pairwise Distance (mm)')
    axes[idx].tick_params(axis='x')

plt.tight_layout()
plt.savefig('violin.svg')
plt.show()

### Raincloud plot - half violin, box, and rainplot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (var, varPlot) in enumerate(zip(groupVars, groupVarNames)):
    # Create raincloud plot
    testVar = pt.RainCloud(data=combodf, x=var, y='mean_distance', ax=axes[idx], 
        palette='Set2', hue=var, bw=0.3, width_viol=0.8, width_box=0.5,
        orient='v', alpha=0.6, dodge=True, pointplot=False, point_size=3, 
        linewidth=2, jitter=0.3)
    
    # Get unique groups
    groups = combodf[var].unique()
    group_data = [combodf[combodf[var] == val]['mean_distance'].dropna()
                  for val in groups]
    
    # Open up the plot to ensure the peaks for each distribution are visible.
    percSpread = 0.4 # Defines how much you want to extend past current limits.
    xmin, xmax = axes[idx].get_xlim()
    axes[idx].set_xlim((xmin-(percSpread), xmax+(percSpread)))

    # Perform statistical test (t-test for 2 groups, ANOVA for more)
    if len(groups) == 2:
        stat, p_val = ttest_ind(group_data[0], group_data[1])
        stat_name = 't'
        
        # Add significance line and star
        y_max = combodf['mean_distance'].max()
        y_min = combodf['mean_distance'].min()
        y_range = y_max - y_min
        line_height = y_max + 0 * y_range
        star_height = line_height + 0.03 * y_range
        
        # Draw bracket line
        axes[idx].plot([0, 1], [line_height, line_height], 'k-', linewidth=1.5)
        axes[idx].plot([0, 0], [line_height - 0.02 * y_range, line_height], 'k-', linewidth=1.5)
        axes[idx].plot([1, 1], [line_height - 0.02 * y_range, line_height], 'k-', linewidth=1.5)
        
        # Add star based on significance level
        if p_val < 0.001:
            star = '***'
        elif p_val < 0.01:
            star = '**'
        elif p_val < 0.05:
            star = '*'
        else:
            star = 'ns'
        
        axes[idx].text(0.5, star_height, star, ha='center', va='bottom', fontsize=16, fontweight='bold')
        
        # Adjust y-axis to accommodate the bracket
        axes[idx].set_ylim(y_min - 0.05 * y_range, star_height + 0.05 * y_range)
        
    else:
        # ANOVA for more than 2 groups
        stat, p_val = f_oneway(*group_data)
        stat_name = 'F'
    
    # Add title with statistics
    axes[idx].set_title(f'{varPlot}\n{stat_name}={stat:.2f}, p={p_val:.4f}',
                        fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('', fontsize=11)
    axes[idx].set_ylabel('Mean Pairwise Distance (mm)', fontsize=11)
    # axes[idx].tick_params(axis='x')

    xmin, xmax = plt.xlim()

plt.tight_layout()
plt.savefig('Rainplot.svg')
plt.show()